In [24]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from scipy.stats import multivariate_normal, norm

snp = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "snp500.csv", index_col=1).reset_index()
china = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "china_transformed.csv", index_col=1).reset_index()
em = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "emerging_markets.csv", index_col=1)
gold = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "gold.csv", index_col=1)
india = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "india.csv", index_col=1)
mscieurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "msci_europe.csv", index_col=1)
smallcapeurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "small_cap_europe.csv", index_col=1)

labels = ["snp", "china", "em", "gold", "india", "mscieurope", "smallcapeurope"]
dfs = [snp, china, em, gold, india, mscieurope, smallcapeurope]

def transform_date(date):
    parts = date.split(' ')
    if len(parts) == 1:
        return date
    return parts[0]

for df in dfs:
    df["Date"] = df["Date"].apply(transform_date)

china

date_sets = [set(df["Date"]) for df in dfs]

# Find the intersection of all date sets
common_dates = set.intersection(*date_sets)

# Filter each DataFrame to include only common dates
filtered_dfs = [df[df["Date"].isin(common_dates)] for df in dfs]

print('length of all dataframes:', [len(df) for df in filtered_dfs])


length of all dataframes: [2651, 2651, 2651, 2651, 2651, 2651, 2651]


In [ ]:
from scipy.stats import rankdata

"""

$Z\sim U(0,\Sigma)$

$X_i=F_{i}^{-1}(\Phi(Z_i))$

use Copula transform to model the joint distribution of X (all assets),
from the distribution of the individual assets

"""

N = len(filtered_dfs)
T = len(filtered_dfs[0]) - 1

returns_all = np.empty((T, N))
sorted_returns_all = np.empty((T, N))

for i, (label, df) in enumerate(zip(labels, filtered_dfs)):
    returns = df["Close"].pct_change().values[1:]
    returns_all[:, i] = returns
    sorted_returns_all[:, i] = sorted(returns)

# Step 1: convert returns → uniforms (rank transform)
# preserves rank dependence
U = np.zeros_like(returns_all) # (T,N)
for i in range(N):
    ranks = rankdata(returns_all[:, i], method='average') # compute ranks to throw away
    # things like units, magnitudes etc
    # for copula correlations: we purely want rank-based information
    U[:, i] = ranks / (T + 1)

def inverse_cdf(u, sorted_returns):
    n = len(sorted_returns)
    idx = np.minimum((u * n).astype(int), n - 1)
    return sorted_returns[idx]
    
Z = norm.ppf(U) # (T, N) (performed elementwise)
# copula covariance: models dependence structure independent of marginals
# dependence is captured in Z
cov_copula = np.cov(Z, rowvar=False) # (N, N)
num_samples = 100

Z_samples = multivariate_normal.rvs(
    mean=np.zeros(N),
    cov=cov_copula,
    size=num_samples
) # shape (num_samples, N)
U_samples = norm.cdf(Z_samples) # (num_samples, N)
X_samples = np.zeros_like(U_samples) # (num_samples, N)

for i in range(N):
    X_samples[:, i] = inverse_cdf(
        U_samples[:, i],
        sorted_returns_all[:, i]
    )



shape Z: (100, 7)
U shape: (100, 7)
X samples: (100, 7)


In [ ]:
"""
optimize weights
"""

import tensorflow as tf

alpha = 0.05   # tail probability for CVaR
lambda_ = 1.0  # CVaR penalty
learning_rate = 0.01
gamma = 0.05
num_steps = 2000
beta = 0.05

# Initialize weights uniformly
w = tf.Variable(tf.ones([N], dtype=tf.float32) / N)
t = tf.Variable(0.0, dtype=tf.float32)  # VaR threshold

optimizer = tf.keras.optimizers.Adam(learning_rate)

In [ ]:
X_tf = tf.convert_to_tensor(X_samples, dtype=tf.float32)
cov = np.cov(returns_all, rowvar=False)
cov_tf = tf.convert_to_tensor(cov, dtype=tf.float32)  # shape (N, N)

for step in range(num_steps):
    with tf.GradientTape() as tape:
        # enforce sum-to-1 via softmax
        w_normalized = tf.nn.softmax(w)

        # portfolio returns
        portfolio_returns = tf.matmul(X_tf, tf.reshape(w_normalized, [-1,1]))
        portfolio_returns = tf.squeeze(portfolio_returns)  # shape: (num_samples,)

        # losses for CVaR (negative returns)
        losses = -portfolio_returns

        # CVaR (Rockafellar–Uryasev)
        cvar = t + (1 / (1 - alpha)) * tf.reduce_mean(tf.nn.relu(losses - t))

        # objective: maximize mean - lambda * CVaR
        # reshape w to column vector
        w_col = tf.reshape(w_normalized, [N, 1])  # shape (N,1)

        # portfolio variance
        portfolio_variance = tf.matmul(tf.matmul(tf.transpose(w_col), cov_tf), w_col)  # shape (1,1)
        portfolio_variance = tf.squeeze(portfolio_variance)  # scalar

        volatility_penalty = beta * portfolio_variance
        diversification_penalty = gamma * tf.reduce_sum(tf.square(w_normalized))
        objective = (
            -(tf.reduce_mean(portfolio_returns) - lambda_ * cvar)
            + diversification_penalty
            + volatility_penalty
        )

    # compute gradients
    grads = tape.gradient(objective, [w, t])
    optimizer.apply_gradients(zip(grads, [w, t]))

    if step % 200 == 0:
        print(f"Step {step}: objective = {-objective.numpy():.6f}")

# after the optimization loop
optimal_w = tf.nn.softmax(w).numpy()
print("Optimal weights:", optimal_w)
print("Sum:", optimal_w.sum())

Step 0: objective = -0.008744
Step 200: objective = -0.006348
Step 400: objective = -0.006348
Step 600: objective = -0.006348
Step 800: objective = -0.006348
Step 1000: objective = -0.006348
Step 1200: objective = -0.006348
Step 1400: objective = -0.006348
Step 1600: objective = -0.006348
Step 1800: objective = -0.006348
Optimal weights: [0.13577801 0.11744606 0.148079   0.18085317 0.15168005 0.12689444
 0.13926923]
Sum: 1.0


In [ ]:

# plot returns

total_return = returns_all @ optimal_w # shape (T,)
cumulative_returns = np.cumprod(total_return + 1) - 1
plot = go.Figure()
plot.add_trace(go.Scatter(x=list(range(T)), y=cumulative_returns))
plot.show()

In [23]:
# compute metrics, given weights
risk_free = 0.0
final_return = cumulative_returns[-1]
sharpe = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * np.std(total_return))
sortino = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * np.std(total_return[total_return > 0]))

alpha = 0.10

var_alpha = np.quantile(total_return, alpha)
cvar = total_return[total_return <= var_alpha].mean()

print('sortino:', sortino)
print('sharpe:', sharpe)
print('cvar:', cvar)




sortino: 1.2043387588811238
sharpe: 0.7459670291558399
cvar: -0.013819071208509155
